<a href="https://colab.research.google.com/github/Tahir-MD/Elevo_NLP_Intership/blob/main/Elevvo_NLP_IIntership_Task_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install sentence-transformers PyPDF2 python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import PyPDF2
import docx
import re
import warnings
warnings.filterwarnings('ignore')

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [4]:
job_descriptions = pd.DataFrame({
    'job_id': [1, 2, 3, 4],
    'job_title': ['Data Scientist', 'Machine Learning Engineer', 'Python Developer', 'Data Analyst'],
    'job_description': [
        'Looking for a Data Scientist with experience in Python, Machine Learning, SQL, and Statistics. Must have strong analytical skills and experience with NLP.',
        'Machine Learning Engineer needed for developing ML models. Required skills: Python, TensorFlow, PyTorch, Deep Learning, MLOps, and Cloud Computing.',
        'Python Developer position. Required: Python, Django, Flask, REST APIs, PostgreSQL, and Git. Experience with AWS is a plus.',
        'Data Analyst role requiring SQL, Excel, Tableau, Python, Data Visualization, and Statistical Analysis. Strong communication skills needed.'
    ]
})

In [5]:
resumes = pd.DataFrame({
    'candidate_id': [101, 102, 103, 104, 105],
    'candidate_name': ['John Smith', 'Sarah Johnson', 'Mike Chen', 'Emily Brown', 'David Wilson'],
    'resume_text': [
        'Data Scientist with 3 years experience. Skilled in Python, Machine Learning, SQL, NLP, and Deep Learning. Worked on sentiment analysis projects using BERT.',
        'Machine Learning Engineer proficient in Python, TensorFlow, PyTorch, and Cloud Computing. Experience in deploying models using AWS SageMaker.',
        'Python Developer skilled in Django, Flask, REST APIs, PostgreSQL, and Git. Built scalable web applications for fintech companies.',
        'Data Analyst with expertise in SQL, Excel, Tableau, and Python. Created dashboards and performed statistical analysis for business decisions.',
        'Software Engineer with Java, Spring Boot, and MySQL experience. Knowledge of basic Python and SQL.'
    ]
})

In [8]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

job_descriptions['clean_description'] = job_descriptions['job_description'].apply(clean_text)
resumes['clean_resume'] = resumes['resume_text'].apply(clean_text)

In [7]:
job_embeddings = model.encode(job_descriptions['clean_description'].tolist(), convert_to_tensor=True)
resume_embeddings = model.encode(resumes['clean_resume'].tolist(), convert_to_tensor=True)
print(f"Embeddings created: {len(job_embeddings)} jobs, {len(resume_embeddings)} resumes")

Embeddings created: 4 jobs, 5 resumes


In [9]:
similarity_matrix = util.cos_sim(resume_embeddings, job_embeddings)
similarity_df = pd.DataFrame(
    similarity_matrix.numpy(),
    index=resumes['candidate_name'],
    columns=job_descriptions['job_title']
)
print("Similarity Matrix:")
print(similarity_df)

Similarity Matrix:
job_title       Data Scientist  Machine Learning Engineer  Python Developer  \
candidate_name                                                                
John Smith            0.700844                   0.491378          0.249393   
Sarah Johnson         0.477524                   0.789297          0.502189   
Mike Chen             0.448642                   0.399257          0.823056   
Emily Brown           0.592915                   0.294647          0.327493   
David Wilson          0.459231                   0.259094          0.442553   

job_title       Data Analyst  
candidate_name                
John Smith          0.437068  
Sarah Johnson       0.353035  
Mike Chen           0.384641  
Emily Brown         0.819365  
David Wilson        0.384039  


In [10]:
common_skills = ['Python', 'SQL', 'Machine Learning', 'Deep Learning', 'NLP', 'TensorFlow', 'PyTorch', 'Django', 'Flask', 'Tableau', 'Excel', 'AWS', 'Git', 'Java', 'Statistics']

def extract_skills(text):
    text = text.lower()
    found = []
    for skill in common_skills:
        if skill.lower() in text:
            found.append(skill)
    return found

resumes['skills'] = resumes['resume_text'].apply(extract_skills)
job_descriptions['required_skills'] = job_descriptions['job_description'].apply(extract_skills)

In [11]:
def rank_resumes(job_title, top_n=3):
    scores = similarity_df[job_title].sort_values(ascending=False).head(top_n)
    results = []
    for candidate, score in scores.items():
        candidate_skills = resumes[resumes['candidate_name'] == candidate].iloc[0]['skills']
        job_skills = job_descriptions[job_descriptions['job_title'] == job_title].iloc[0]['required_skills']
        matched = list(set(job_skills).intersection(set(candidate_skills)))
        results.append({
            'rank': len(results) + 1,
            'candidate': candidate,
            'score': f"{score*100:.2f}%",
            'matched_skills': matched
        })
    return pd.DataFrame(results)

print("Top candidates for Data Scientist:")
print(rank_resumes('Data Scientist'))

Top candidates for Data Scientist:
   rank      candidate   score                        matched_skills
0     1     John Smith  70.08%  [SQL, Machine Learning, Python, NLP]
1     2    Emily Brown  59.29%                         [SQL, Python]
2     3  Sarah Johnson  47.75%            [Machine Learning, Python]


In [12]:
print("\nTop candidates for Python Developer:")
print(rank_resumes('Python Developer'))


Top candidates for Python Developer:
   rank      candidate   score                     matched_skills
0     1      Mike Chen  82.31%  [SQL, Python, Git, Django, Flask]
1     2  Sarah Johnson  50.22%                      [Python, AWS]
2     3   David Wilson  44.26%                      [SQL, Python]


In [13]:
def recommend_jobs(candidate_name, top_n=3):
    scores = similarity_df.loc[candidate_name].sort_values(ascending=False).head(top_n)
    print(f"\nRecommended jobs for {candidate_name}:")
    for job, score in scores.items():
        job_skills = job_descriptions[job_descriptions['job_title'] == job].iloc[0]['required_skills']
        candidate_skills = resumes[resumes['candidate_name'] == candidate_name].iloc[0]['skills']
        matched = list(set(job_skills).intersection(set(candidate_skills)))
        print(f"{job}: {score*100:.2f}% match, Skills matched: {matched}")

recommend_jobs('John Smith')
recommend_jobs('Mike Chen')


Recommended jobs for John Smith:
Data Scientist: 70.08% match, Skills matched: ['SQL', 'Machine Learning', 'Python', 'NLP']
Machine Learning Engineer: 49.14% match, Skills matched: ['Machine Learning', 'Python', 'Deep Learning']
Data Analyst: 43.71% match, Skills matched: ['SQL', 'Python']

Recommended jobs for Mike Chen:
Python Developer: 82.31% match, Skills matched: ['SQL', 'Python', 'Git', 'Django', 'Flask']
Data Scientist: 44.86% match, Skills matched: ['SQL', 'Python']
Machine Learning Engineer: 39.93% match, Skills matched: ['Python']


In [14]:
all_matches = []
for candidate in resumes['candidate_name']:
    for job in job_descriptions['job_title']:
        score = similarity_df.loc[candidate, job]
        all_matches.append({
            'candidate': candidate,
            'job': job,
            'match_score': f"{score*100:.2f}%",
            'numeric_score': score
        })

results_df = pd.DataFrame(all_matches)
best_match = results_df.loc[results_df['numeric_score'].idxmax()]
print(f"Best overall match: {best_match['candidate']} for {best_match['job']} with {best_match['match_score']}")

Best overall match: Mike Chen for Python Developer with 82.31%


In [15]:
results_df.to_csv('resume_screening_results.csv', index=False)
print("Results saved to resume_screening_results.csv")

print("\n" + "="*50)
print("RESUME SCREENING SUMMARY")
print("="*50)
print(f"Jobs: {len(job_descriptions)}, Candidates: {len(resumes)}")
print(f"Total matches evaluated: {len(results_df)}")

Results saved to resume_screening_results.csv

RESUME SCREENING SUMMARY
Jobs: 4, Candidates: 5
Total matches evaluated: 20


In [16]:
def quick_match(candidate_name, job_title):
    score = similarity_df.loc[candidate_name, job_title]
    candidate_skills = resumes[resumes['candidate_name'] == candidate_name].iloc[0]['skills']
    job_skills = job_descriptions[job_descriptions['job_title'] == job_title].iloc[0]['required_skills']
    matched = list(set(job_skills).intersection(set(candidate_skills)))
    missing = list(set(job_skills) - set(candidate_skills))

    print(f"\n{'='*50}")
    print(f"MATCH REPORT: {candidate_name} for {job_title}")
    print(f"{'='*50}")
    print(f"Match Score: {score*100:.2f}%")
    print(f"✓ Skills matched: {matched if matched else 'None'}")
    print(f"✗ Skills missing: {missing if missing else 'None'}")
    print(f"Verdict: {'Strong Match' if score > 0.7 else 'Good Match' if score > 0.5 else 'Weak Match'}")

quick_match('Sarah Johnson', 'Machine Learning Engineer')
quick_match('David Wilson', 'Data Analyst')


MATCH REPORT: Sarah Johnson for Machine Learning Engineer
Match Score: 78.93%
✓ Skills matched: ['Machine Learning', 'PyTorch', 'TensorFlow', 'Python']
✗ Skills missing: ['Deep Learning']
Verdict: Strong Match

MATCH REPORT: David Wilson for Data Analyst
Match Score: 38.40%
✓ Skills matched: ['SQL', 'Python']
✗ Skills missing: ['Tableau', 'Excel']
Verdict: Weak Match
